# Le crâne de Jussac

Jussac, Auvergne. Le crâne d’un animal préhistorique est découvert. De la famille des canidés, les scientifiques se demandent s’il tenait plus du chien ou du loup. Saurez-vous répondre à la question ?

Votre analyse en composantes principales ne devra reposer que sur les outils ci-dessous :

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Présentation des données

Le jeu de données est constitué de 43 individus, dont 30 chiens et 12 loups, décrits à l’aide de six caractéristiques physiques :

|Variable|Description|Unité|
|:-:|:-|:-:|
|*LCB*|Longueur condylo-basale|mm|
|*LMS*|Longueur de la mâchoire supérieure|mm|
|*LBM*|Largeur bi-maxilaire|mm|
|*LP*|Longueur de la carnassière supérieure|mm|
|*LM*|Longueur de la première molaire supérieure|mm|
|*LAM*|Largeur de la première molaire supérieure|mm|
|*Espèce*|Facteur à deux niveaux : `chien` ou `loup`|-|

> Bonis, Louis & Lebeaux, M.-O. (1974). — À propos d'un crâne de Canidé découvert dans le Quaternaire du Cantal : étude multidimensionnelle des caractères discriminants entre chiens et loups. *Mammalia* vol. 38, n° 4, p. 717-728. https://doi.org/10.1515/mamm.1974.38.4.717.

## Préparation des données

Commencez par charger [les données](../data/jussac.csv) dans un *data frame*, sachant que le séparateur est la tabulation :

In [ ]:
# your code here

import pandas as pd

df = pd.read_csv('../data/jussac.csv', delimiter='\t')

Dans une variable `data`, centrez et réduisez les variables :

In [ ]:
# your code here

import numpy as np

data = df.copy().drop(columns='Espèce')
data = data.apply(lambda x: (x - np.mean(x)) / np.std(x))

## Décomposition des données en composantes principales

### La matrice de covariance

Produisez une matrice de covariance :

In [ ]:
# your code here

cov_matrix = np.cov(data, rowvar=False)

Vérifiez que vous disposez bien d’une matrice carrée de taille $(6, 6)$ :

In [ ]:
# your code here

cov_matrix.shape

### La décomposition en vecteurs et valeurs propres

Décomposez la matrice de covariance en vecteurs et valeurs propres :

In [ ]:
# your code here

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

Vérifiez à présent le taux de variance expliquée par chaque composante principale :

In [ ]:
# your code here

for n, eigenvalue in enumerate(eigenvalues):
    print(f"PC{n + 1} explique{eigenvalue / sum(eigenvalues): .2%} de la variance")

Vous devriez trouver que la dernière composante principale, PC6, en révèle davantage que la précédente. Rétablissez l’ordre décroissant :

In [ ]:
# your code here

idx = eigenvalues.argsort()[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

### Détermination du nombre de facteurs

Affichez le diagramme d’éboulis :

In [ ]:
_ = sns.lineplot(x=range(1, len(eigenvalues) + 1), y=eigenvalues)
plt.title("Diagramme d’éboulis")
plt.xlabel("Numéro de la composante principale")
plt.ylabel("Explication de la variance")
sns.despine()

Si le diagramme d’éboulis vous suffit pour choisir le nombre de composantes principales à charger ensuite dans le jeu de données, passez à l’étape de réduction de la dimension. Autrement, testez d’autres méthodes :

In [ ]:
# your code here

### Réduction de la dimensionnalité

Sélectionnez la matrice des composantes principales, limitées à celles que vous avez sélectionnées à l’étape précédente, et chargez-la dans le jeu de données centrées-réduites :

In [ ]:
# your code here

# nb of components
n_components = 2

# matrix multiplication
pca = data.dot(eigenvectors[:, :n_components])

# rename columns
pca.columns = [ f"PC{n}" for n in range(1, n_components + 1) ]

# insert variable 'Espèce'
pca = pd.concat([pca, df.Espèce], axis=1)

## Produire une analyse

### Analyse à l’aide d’un graphique

En vous aidant des modes de représentations que vous connaissez, à savoir le cercle des corrélations et/ou le nuage de points, déterminez si l’individu n°42 était un chien ou un loup :

In [ ]:
# your code here

pca.Espèce = pca.Espèce.fillna("inconnue")

palette = ['tab:orange', 'tab:olive', 'tab:blue']

fig = sns.scatterplot(data=pca, x="PC1", y="PC2", hue="Espèce", palette=palette)
fig.set_title("Le crâne de Jussac : chien ou loup ?")
fig.set_xlabel("PC1 (68,33 %)")
fig.set_ylabel("PC2 (14.71 %)")

sns.despine()

### Classification à l’aide d’un modèle mathématique

Commençons par isoler l’enregistrement à étiqueter et à travailler sur une copie des données :

In [ ]:
point = np.array([0.249249, -0.020491])
df = pca.drop(42).copy()

#### Clustering

Si nous suivons l’hypothèse que les caractéristiques physiologiques mesurées dans l’article de Louis Bonis et M.-O. Lebeaux permettent de séparer nettement les chiens des loups, on peut commencer par considérer deux clusters dont on va déterminer les centroïdes. La méthode consiste à calculer la moyenne des coordonnées de chaque classe avant de calculer la distance euclidienne des coordonnées du crâne inconnu avec chaque centroïde.

Calculons tout d’abord les centroïdes de chaque classe :

In [ ]:
centroids = df.groupby('Espèce')[['PC1', 'PC2']].mean()

Et maintenant la distance du point à classer avec chaque centroïde :

In [ ]:
dist_dog = np.linalg.norm(centroids.loc["chien"] - point)
dist_wolf = np.linalg.norm(centroids.loc["loup"] - point)

Selon cette méthode, le crâne inconnu serait un chien.

#### Les plus proches voisins

La méthode des *k* plus proches voisins repose sur une intuition simple : l’enregistrement à étiqueter appartient sans doute à la classe des enregistrements autour de lui. Pour cela, on considère *k* voisins les plus proches afin d’opter pour le vote majoritaire.

Commençons par définir le nombre de voisins et par constituer les données d’apprentissage :

In [ ]:
k = 5

X = df[['PC1', 'PC2']].to_numpy()
y = df['Espèce'].to_numpy()

Calculons à présent les distances euclidiennes entre chaque enregistrement et le point à classer :

In [ ]:
distances = np.linalg.norm(X - point, axis=1)

De là, nous pouvons cerner les indices des *k* plus proches voisins avec la méthode `.argsort()` :

In [ ]:
nearest_idx = distances.argsort()[:5]

Les indices nous permettent de retrouver les étiquettes correspondantes :

In [ ]:
labels = y[nearest_idx]

Il reste à les dénombrer et à retenir l’étiquette majoritaire :

In [ ]:
counts = np.unique(labels, return_counts=True)
animal = counts[0][np.argmax(counts[1])]

Là encore, la méthode nous indique que le crâne devait appartenir à un chien.

#### La régression logistique

La régression logistique est une méthode de classification binaire qui prédit la probabilité qu’un individu appartienne à une classe. Elle utilise une fonction sigmoïde pour transformer une combinaison linéaire des variables en valeur comprise entre 0 et 1. On attribue ensuite la classe en fonction d’un seuil (souvent 0,5). Cette méthode permet d’obtenir à la fois la classe prédite et la probabilité d’appartenance.

Le modèle s’exprime par la formule :

$$
P(y = 1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

Où :

$$
z = w_0 + w_1 x_1 + w_2 x_2
$$

Définissons tout d’abord la fonction sigmoïde chargée de transformer la prédiction en probabilité :

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

Prenons ensuite les données sur lesquelles travailler :

In [ ]:
X = df[['PC1', 'PC2']].to_numpy()
y = np.array([1 if label=='chien' else 0 for label in df['Espèce']])

Pour inclure le terme de biais $w_0$ dans le modèle, on ajoute une colonne de 1 à nos données. Elle permet à la frontière de décision de ne pas être obligatoirement centrée à l’origine et rend ainsi le modèle plus flexible :

In [ ]:
point_aug = np.hstack([1, point]).reshape(1, -1)
X_aug = np.hstack([np.ones((X.shape[0], 1)), X])

Il reste à apprendre les poids via la descente de gradient. On initialise en premier lieu les poids à zéro tout en définissant un taux d’apprentissage et le nombre d’itérations :

In [ ]:
w = np.zeros(X_aug.shape[1])
lr = 0.1
n_iter = 1000

À chaque itération, les poids se mettront à jour à partir des données :

In [ ]:
for _ in range(n_iter):
    # Compute the linear combination of inputs and weights: z = w0 + w1*x1 + w2*x2
    z = X_aug @ w
    # Apply the sigmoid function to get predicted probabilities
    y_pred = sigmoid(z)
    # Compute the gradient of the log-loss with respect to the weights (average over all samples)
    grad = X_aug.T @ (y_pred - y) / len(y)
    # Update the weights in the direction that reduces the loss
    w -= lr * grad

On applique enfin la fonction sigmoïde sur le produit matriciel entre le vecteur du point à classer et les poids appris pour obtenir la probabilité qu’il appartienne à la classe **chien** :

In [ ]:
prob = sigmoid(point_aug @ w)[0]

# Threshold at 0.5
animal = 'chien' if prob >= 0.5 else 'loup'

print(f"Le crâne appartient à la classe {animal}, selon une probabilité calculée d’être un chien de {prob:.2%}")